# YOLO11 Object Detection — Dog & Cat Dataset

This notebook demonstrates training a **YOLO11** model using the [Ultralytics](https://github.com/ultralytics/ultralytics) library on a custom **Dog & Cat** dataset downloaded from Roboflow.

**Workflow:**
1. Prepare Dataset (download via Roboflow)
2. Train the YOLO11 model
3. Evaluate and run inference

> **Prerequisites:** Set your Roboflow API key as an environment variable:  
> `export ROBOFLOW_API_KEY="your_key_here"` (Linux/macOS)  
> `$env:ROBOFLOW_API_KEY = "your_key_here"` (Windows PowerShell)

In [ ]:
# ── Step 1: Install required libraries ──────────────────────────────────────
!pip install roboflow ultralytics --quiet

In [ ]:
# ── Step 2: Download Dataset from Roboflow ───────────────────────────────────
# The API key is loaded from an environment variable to keep it secure.
# Set it before running: export ROBOFLOW_API_KEY="your_key_here"

import os
from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "ROBOFLOW_API_KEY environment variable is not set. "
        "Please set it before running this cell."
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace("appledetect").project("dog_cat_objects")
version = project.version(1)
dataset = version.download("yolov5")  # Downloads into ./dog_cat_objects-1/

# Save the dataset path for later use
DATA_YAML = os.path.join(dataset.location, "data.yaml")
print(f"Dataset downloaded to: {dataset.location}")
print(f"data.yaml path: {DATA_YAML}")

In [ ]:
# ── Step 3: Train YOLO11 Model ───────────────────────────────────────────────
from ultralytics import YOLO

# Load YOLO11 nano model (pre-trained on COCO)
model = YOLO("yolo11n.pt")

# Train on the custom Dog & Cat dataset
results = model.train(
    data=DATA_YAML,   # Path to data.yaml from the downloaded dataset
    epochs=5,          # Number of training epochs
    imgsz=640,         # Input image size
    batch=16,          # Batch size (reduce if out of memory)
    device="cpu",      # Use 'cuda:0' if a GPU is available
    name="yolo11_dog_cat"  # Name of the training run
)

print(f"\nBest model saved to: {results.save_dir}/weights/best.pt")

In [ ]:
# ── Step 4: Evaluate the Trained Model ──────────────────────────────────────
from ultralytics import YOLO
import os

# Load the best weights from the training run
best_model_path = os.path.join(results.save_dir, "weights", "best.pt")
model = YOLO(best_model_path)

# Run validation on the validation split
metrics = model.val(data=DATA_YAML)

print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

In [ ]:
# ── Step 5: Run Inference on Test Images ─────────────────────────────────────
import os
from pathlib import Path

# Folder containing test images (update this path to your test image directory)
image_folder = Path(dataset.location) / "test" / "images"
output_folder = Path("outputs") / "inference_results"
output_folder.mkdir(parents=True, exist_ok=True)

# Run batch inference
results_list = model.predict(
    source=str(image_folder),
    save=True,                    # Save annotated images
    project=str(output_folder),   # Output directory
    name="predict",
    conf=0.25,                    # Confidence threshold
    exist_ok=True
)

print(f"\nInference complete. Results saved to: {output_folder}/predict")